# Measurements and Limits

A model used for operations, such as by a state estimator, a DMS or GridAPPS-D, needs to say what is
measured where, and the limits that equipment must stay within. `AnalogBuilder` and `DiscreteBuilder` make
the measurements, and `OperationalLimitSetBuilder` the limits.

## Set up

The CIM profile and unit handling are process-wide environment variables, so set them before importing
anything from CIMantic Graphs. `new_feeder_model` makes an empty model to build into.

In [1]:
import os

os.environ["CIMG_CIM_PROFILE"] = "cim18v17"
os.environ["CIMG_USE_UNITS"] = "TRUE"
os.environ["CIMG_SYSTEM_MVA_BASE"] = "100"

from cimtables.units import Qty

from cimbuilder.network import new_feeder_model
from cimbuilder.object_builder import (
    ACLineSegmentBuilder,
    AnalogBuilder,
    BaseVoltageBuilder,
    BreakerBuilder,
    DiscreteBuilder,
    EnergyConsumerBuilder,
    OperationalLimitSetBuilder,
)

network = new_feeder_model(
    authority="https://example.com/models",
    model="measurements_and_limits",
    Feeder="fdr_demo",
)
container = network.container
base_voltage_builder = BaseVoltageBuilder(network, container)
for mRID, nominalVoltage in [
    ("bv_12_47kV", Qty(12.47, "kV")),
]:
    base_voltage_builder.create(mRID=mRID, name=mRID, nominalVoltage=nominalVoltage)

## The equipment to measure

A breaker, a line and a load in series:

In [2]:
breaker_builder = BreakerBuilder(network, container)
breaker = breaker_builder.create(mRID="brk_1", name="feeder breaker")
breaker_builder.add_connectivity(
    breaker, node1="subbus", node2="head", phases="ABC", BaseVoltage="bv_12_47kV"
)

line_builder = ACLineSegmentBuilder(network, container)
line = line_builder.create(mRID="line_1", name="line 1")
line_builder.add_connectivity(
    line,
    node1="head",
    node2="bus_1",
    phases="ABC",
    length=Qty(1, "mi"),
    BaseVoltage="bv_12_47kV",
)

load_builder = EnergyConsumerBuilder(network, container)
load = load_builder.create(mRID="ld_1", name="load 1")
load_builder.add_connectivity(
    load, node="bus_1", phases="ABC", BaseVoltage="bv_12_47kV"
)

for equipment in (breaker, line, load):
    terminals = sorted(
        equipment.Terminals, key=lambda terminal: terminal.sequenceNumber
    )
    print(equipment.mRID, [terminal.ConnectivityNode.mRID for terminal in terminals])

brk_1 ['subbus', 'head']
line_1 ['head', 'bus_1']
ld_1 ['bus_1']


## Standard measurements per piece of equipment

Each `add_<equipment>_analogs` method makes the usual `Analog`s for one piece of equipment, one per phase,
with the IEC 62361-102 `measurementType` and the IEC 61850 name as `description`:

| Method | Measures |
|---|---|
| `add_load_analogs`, `add_shunt_analogs`, `add_generator_analogs` | voltage and power at the Terminal |
| `add_inverter_analogs` | voltage and power at the Terminal, and the state of charge of each battery |
| `add_line_analogs` | power at Terminal 1, and voltage there if the node has none yet |
| `add_switch_analogs` | current at Terminal 1, and voltage at both Terminals where the nodes have none yet |
| `add_transformer_analogs` | power at each end, and voltage where the nodes have none yet (also regulators) |
| `add_tap_changer_analogs` | the tap position |

`DiscreteBuilder.add_switch_discretes` and `add_shunt_discretes` make the switch positions. Each method
returns the equipment and its measurements, and calling it again returns the same ones rather than adding
duplicates.

In [3]:
analog_builder = AnalogBuilder(network, container)
discrete_builder = DiscreteBuilder(network, container)

_, load_analogs = analog_builder.add_load_analogs(load)
_, line_analogs = analog_builder.add_line_analogs(line)
_, breaker_analogs = analog_builder.add_switch_analogs(breaker)
_, positions = discrete_builder.add_switch_discretes(breaker)

for measurement in load_analogs + line_analogs + breaker_analogs + positions:
    kind = type(measurement).__name__
    phase = measurement.phases.value
    terminal = measurement.Terminal.mRID
    row = f"{kind:9} {measurement.measurementType:22} {phase} {terminal:10}"
    print(row, measurement.description)

Analog    PhaseToNeutralVoltage  A ld_1_T1    MMXU.PNV.phsA
Analog    PhaseToNeutralVoltage  B ld_1_T1    MMXU.PNV.phsB
Analog    PhaseToNeutralVoltage  C ld_1_T1    MMXU.PNV.phsC
Analog    ApparentPower          A ld_1_T1    MMXU.VA.phsA
Analog    ApparentPower          B ld_1_T1    MMXU.VA.phsB
Analog    ApparentPower          C ld_1_T1    MMXU.VA.phsC
Analog    ApparentPower          A line_1_T1  MMXU.VA.phsA
Analog    ApparentPower          B line_1_T1  MMXU.VA.phsB
Analog    ApparentPower          C line_1_T1  MMXU.VA.phsC
Analog    PhaseToNeutralVoltage  A line_1_T1  MMXU.PNV.phsA
Analog    PhaseToNeutralVoltage  B line_1_T1  MMXU.PNV.phsB
Analog    PhaseToNeutralVoltage  C line_1_T1  MMXU.PNV.phsC
Analog    PhaseCurrent           A brk_1_T1   MMXU.A.phsA
Analog    PhaseCurrent           B brk_1_T1   MMXU.A.phsB
Analog    PhaseCurrent           C brk_1_T1   MMXU.A.phsC
Analog    PhaseToNeutralVoltage  A brk_1_T1   MMXU.PNV.phsA
Analog    PhaseToNeutralVoltage  B brk_1_T1   MMXU.P

The breaker measured voltage only at Terminal 1, on `subbus`: its Terminal 2 node, `head`, already had
the line's voltage measurements. Measurement mRIDs are UUIDs seeded from the equipment, so a rebuilt model gets the same ones.

## One measurement by hand

`create` and `add_measurement` make any other measurement:

In [4]:
analog = analog_builder.create(
    mRID="ld_1_pf", name="ld_1 power factor", description="MMXU.TotPF"
)
analog_builder.add_measurement(
    analog,
    measurementType="PowerFactor",
    PowerSystemResource="ld_1",
    Terminal="ld_1_T1",
    unitSymbol="none",
)
print(analog.measurementType, analog.PowerSystemResource.mRID, analog.Terminal.mRID)

PowerFactor ld_1 ld_1_T1


## Operational limits

An `OperationalLimitSet` groups the limits of one piece of equipment, optionally at one of its Terminals.
Each limit value has an `OperationalLimitType` saying how long it may be exceeded: here a normal rating,
held indefinitely, and a four-hour emergency rating.

In [5]:
limit_builder = OperationalLimitSetBuilder(network, container)
limit_builder.add_limit_type(
    mRID="olt_normal",
    name="normal",
    isInfiniteDuration=True,
    direction="absoluteValue",
)
limit_builder.add_limit_type(
    mRID="olt_emergency",
    name="emergency",
    acceptableDuration=Qty(4, "hour"),
    direction="absoluteValue",
)

limit_set = limit_builder.create(mRID="line_1_limits", name="line 1 ampacity")
limit_builder.add_equipment(limit_set, Equipment="line_1", terminal_sequence_number=1)
limit_builder.add_current_limit(
    limit_set, value=Qty(530, "A"), OperationalLimitType="olt_normal"
)
limit_builder.add_current_limit(
    limit_set, value=Qty(660, "A"), OperationalLimitType="olt_emergency"
)

for limit in limit_set.OperationalLimitValue:
    print(
        f"{limit.mRID}: {limit.OperationalLimitType.name}, {limit.value.to('A'):g} A"
    )

line_1_limits_current_limit_olt_normal: normal, 530 A
line_1_limits_current_limit_olt_emergency: emergency, 660 A


`add_voltage_limit`, `add_active_power_limit` and `add_apparent_power_limit` add the other kinds of limit
the same way.